In [ ]:
import sys, os, importlib, yaml
from unittest.mock import Mock, patch

import corgihowfsc

mode = "spec_band3"  # try: nfov_band1, wfov_band1, wfov_band4, spec_band2, spec_band3, specrot_band2, specrot_band3

# Mock only the heavy corgisim optical package. cgisim is used for real:
# _get_corgisim_subbands() reads the authoritative subband wavelength ranges
# via cgisim.cgisim_read_bandpass, so cgisim must be importable in this env.
fake_sysmods = {
    "corgisim": Mock(),
    "corgisim.scene": Mock(),
    "corgisim.instrument": Mock(),
}

import logging
logging.basicConfig(level=logging.DEBUG, format="%(levelname)s %(name)s: %(message)s", force=True)


In [ ]:
with patch.dict(sys.modules, fake_sysmods):
    manager_mod = importlib.reload(importlib.import_module("corgihowfsc.utils.corgisim_manager"))
    CorgisimManager = manager_mod.CorgisimManager

    howfscpath = os.path.dirname(os.path.abspath(corgihowfsc.__file__))
    cfgfile = os.path.join(howfscpath, "model", mode, f"{mode}_both_sides", "howfsc_optical_model.yaml")
    channels = yaml.safe_load(open(cfgfile))["sls"]

    cfg = Mock()
    cfg.sl_list = [Mock(lam=channels[i]["lam"]) for i in sorted(channels)]

    manager = CorgisimManager(
        cfg, Mock(), {"star": {"stellar_vmag": 2.5, "stellar_type": "G2V"}}, cor=mode
    )

    # --- what got set up once, at init ---
    print("self.bandpass  :", manager.bandpass)      # band number, from the MIDDLE channel's subband
    print("self.cor_mapped:", manager.cor_mapped)
    print()

    # --- what happens per channel, on create_optics() ---
    dm = __import__("numpy").zeros((48, 48))
    for lind, sl in enumerate(cfg.sl_list):
        wl_nm = sl.lam * 1e9
        try:
            recipe = manager._get_bandpass_recipe(lind)
            print(f"channel {lind}: {wl_nm:.1f} nm -> recipe = {recipe!r}")
        except ValueError as e:
            print(f"channel {lind}: {wl_nm:.1f} nm -> RAISES: {e}")


In [ ]:
optics = manager.create_optics(dm, dm, 0)